# Inicio

## Requirements

In [ ]:
# ============================================================
# Instalación de librerías necesarias desde el notebook
# ============================================================
# import importlib.util
# import subprocess
# import sys

# REQUIRED_PACKAGES = {
#     "numpy": "numpy",
#     "ydata-profiling": "ydata-profiling",
#     "scikit-surprise": "scikit-surprise",
#     # "rarfile": "rarfile",
# }

# missing = [pip_name for import_name, pip_name in REQUIRED_PACKAGES.items()
#            if importlib.util.find_spec(import_name) is None]

# if missing:
#     print("Instalando librerías faltantes:", missing)
#     subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
# else:
#     print("Todas las librerías necesarias ya están instaladas.")

Instalando librerías faltantes: ['ydata-profiling', 'scikit-surprise']


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import gdown
file_id='10onBXp0Krg7RODnT2q2JYSBV_O9HI2ZS'
file_url = f'https://drive.google.com/uc?id={file_id}'
output_path = 'requirements.txt'
gdown.download(file_url, output_path, quiet=False)

Downloading...
From: https://drive.google.com/uc?id=10onBXp0Krg7RODnT2q2JYSBV_O9HI2ZS
To: /content/requirements.txt
100%|██████████| 52.0/52.0 [00:00<00:00, 65.8kB/s]


'requirements.txt'

In [3]:
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.4/154.4 kB 19.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 127.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.8/400.8 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 106.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.7/679.7 kB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 8.1 MB/s eta 0:00:00
  Created wheel for scikit-surprise: filename=scikit_surprise-1.1.4-cp312-cp312-linux_x86_64.whl size=2554986 sha256=0c1e9c2fda2c1616ede13e717ad

## Imports

In [1]:
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

from surprise import SVD, SVDpp
from surprise import Reader
from surprise import Dataset
from surprise import accuracy
from surprise.model_selection import GridSearchCV, train_test_split

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import make_scorer, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics.pairwise import cosine_similarity

from concurrent.futures import ThreadPoolExecutor

import json
import time
import os
import random
import warnings
import duckdb
import multiprocessing
import joblib
from datetime import datetime

from ydata_profiling import ProfileReport # Perfilamiento de datos

%matplotlib inline

/tmp/ipykernel_1132/1740717025.py:30: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport # Perfilamiento de datos


In [3]:
# Para garantizar reproducibilidad en resultados, se define la semilla global
seed = 10
random.seed(seed)
np.random.seed(seed)

# Configuración global
warnings.filterwarnings('ignore')
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option('display.float_format', '{:.3f}'.format) # Configuración global de presentación de .3 decimales en resul

def timer(start_time=None):
  if not start_time:
    start_time = datetime.now()
    return start_time
  elif start_time:
    thour, temp_sec = divmod((datetime.now() - start_time).total_seconds(), 3600)
    tmin, tsec = divmod(temp_sec, 60)
    print('\n Time taken: %i hour(s) %i minute(s) and %s second(s).' % (thour, tmin, round(tsec, 2)))

# Carga de datos

## Business

In [ ]:
business_path='/content/drive/MyDrive/Maestria Andes/Sistemas_recomendacion__/Taller 2/yelp_academic_dataset_business.json'
df_business = pd.read_json(business_path, lines=True)
print(df_business.shape)
df_business.head()

(150346, 14)


,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.427,-119.711,5.000,7,0,{'ByAppointmentOnly': 'True'},"Doctors, Traditional Chinese Medicine, Naturop...",None
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,87 Grasso Plaza Shopping Center,Affton,MO,63123,38.551,-90.336,3.000,15,1,{'BusinessAcceptsCreditCards': 'True'},"Shipping Centers, Local Services, Notaries, Ma...","{'Monday': '0:0-0:0', 'Tuesday': '8:0-18:30', ..."
2,tUFrWirKiKi_TAnsVWINQQ,Target,5255 E Broadway Blvd,Tucson,AZ,85711,32.223,-110.880,3.500,22,0,"{'BikeParking': 'True', 'BusinessAcceptsCredit...","Department Stores, Shopping, Fashion, Home & G...","{'Monday': '8:0-22:0', 'Tuesday': '8:0-22:0', ..."
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.956,-75.156,4.000,80,1,"{'RestaurantsDelivery': 'False', 'OutdoorSeati...","Restaurants, Food, Bubble Tea, Coffee & Tea, B...","{'Monday': '7:0-20:0', 'Tuesday': '7:0-20:0', ..."
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,101 Walnut St,Green Lane,PA,18054,40.338,-75.472,4.500,13,1,"{'BusinessAcceptsCreditCards': 'True', 'Wheelc...","Brewpubs, Breweries, Food","{'Wednesday': '14:0-22:0', 'Thursday': '16:0-2..."


## Users

In [4]:
user_path='/content/drive/MyDrive/Maestria Andes/Sistemas_recomendacion__/Taller 2/yelp_academic_dataset_user.json'
df_user = pd.read_json(user_path, lines=True)
print(df_user.shape)
df_user.head()

(1987897, 22)


,user_id,name,review_count,yelping_since,useful,funny,cool,elite,friends,fans,average_stars,compliment_hot,compliment_more,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos
0,qVc8ODYU5SZjKXVBgXdI7w,Walker,585,2007-01-25 16:47:26,7217,1259,5994,2007,"NSCy54eWehBJyZdG2iE84w, pe42u7DcCH2QmI81NX-8qA...",267,3.910,250,65,55,56,18,232,844,467,467,239,180
1,j14WgRoU_-2ZE1aw1dXrJg,Daniel,4333,2009-01-25 04:35:42,43091,13066,27281,"2009,2010,2011,2012,2013,2014,2015,2016,2017,2...","ueRPE0CX75ePGMqOFVj6IQ, 52oH4DrRvzzl8wh5UXyU0A...",3138,3.740,1145,264,184,157,251,1847,7054,3131,3131,1521,1946
2,2WnXYQFK0hXEoTxPtV2zvg,Steph,665,2008-07-25 10:41:00,2086,1010,1003,"2009,2010,2011,2012,2013","LuO3Bn4f3rlhyHIaNfTlnA, j9B4XdHUhDfTKVecyWQgyA...",52,3.320,89,13,10,17,3,66,96,119,119,35,18
3,SZDeASXq7o05mMNLshsdIA,Gwen,224,2005-11-29 04:38:33,512,330,299,"2009,2010,2011","enx1vVPnfdNUdPho6PH_wg, 4wOcvMLtU6a9Lslggq74Vg...",28,4.270,24,4,1,6,2,12,16,26,26,10,9
4,hA5lMy-EnncsH4JoR-hFGQ,Karen,79,2007-01-05 19:40:59,29,15,7,,"PBK4q9KEEBHhFvSXCUirIw, 3FWPpM7KU1gXeOM_ZbYMbA...",1,3.540,1,1,0,0,0,1,1,0,0,0,0


In [5]:
df_user_sampled=df_user.sample(frac=0.7, random_state=seed)
df_user_sampled.shape

(1391528, 22)

## Reviews

In [ ]:
# review_path='/content/drive/MyDrive/Maestria Andes/Sistemas_recomendacion__/Taller 2/yelp_academic_dataset_review.json'
# df_review = pd.read_json(review_path, lines=True)
# print(df_review.shape)
# df_review.head()

In [6]:
path='/content/drive/MyDrive/Maestria Andes/Sistemas_recomendacion__/Taller 2/yelp_academic_dataset_review.json'

# Only load target users
target_users=df_user_sampled['user_id'].drop_duplicates().tolist()
pd.DataFrame({"user_id": target_users}).to_csv("target_users.csv", index=False)

query = f"""
COPY (
    SELECT r.*
    FROM read_json_auto('{path}') r
    INNER JOIN read_csv_auto('target_users.csv') u
    ON r.user_id = u.user_id
)
TO 'filtered_reviews.jsonl'
(FORMAT JSON);
"""
duckdb.sql(query)

df_review = pd.read_json("filtered_reviews.jsonl", lines=True)
print(df_review.shape)
df_review.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(4900385, 9)


,review_id,user_id,business_id,stars,useful,funny,cool,text,date
0,Ub80H8C5mTHb5FkMsN8VbA,v6bKaR7Hyt9dfkX5dsldSQ,UCMSWPqzXjd7QHq7v8PJjQ,5,1,0,1,Excellent brunch place...could be in LA or San...,2018-02-25 03:13:30
1,aU0q9u4owcy3MTxDqs8Low,C6f720G4P2fV067i3j3XQg,1Pxg1AMf0rEn9QF__ZYoWw,5,0,0,0,The sushi was AMAZING! We got great service a...,2014-06-28 23:55:39
2,0B85NOKEMZR3gPyuG7c7bQ,yYASryt2cwZz3olM-iJT7Q,wI51ie-6j7y5MzxOCS4fNA,5,0,0,0,I'm a long time fan of Tangelo's. The food is ...,2017-01-03 17:40:10
3,-kL5es9sapkDpgTMeI_09Q,X7JH2HXek83O9AuMD_suFw,x-O0dIeIVaVBEhTu_w56DQ,3,0,0,0,better than coco's key. it gets very crowded ...,2016-01-17 22:27:54
4,wYeAU9jINzzA4eg0SDoYwQ,CSx-cOiyUdsjgbe7cqOLbg,RJPRi1pwocHNZr9ISz_P-A,2,3,0,0,"To put it as nicely as I can, everything about...",2015-08-20 03:08:28


## Profiling

In [ ]:
# Perfilamiento
profiling=0
if profiling:
  sample=.1
  output_path='/content/drive/MyDrive/Maestria Andes/Sistemas_recomendacion__/Taller 2/'

  #Business
  df_sampled=df_business.sample(frac=sample, random_state=seed)
  profile_checkin = ProfileReport(df_sampled)
  profile_checkin.to_file(output_file=output_path+'profile_business.html')
  # #User
  df_sampled=df_user.sample(frac=sample/10, random_state=seed)
  profile_checkin = ProfileReport(df_sampled)
  profile_checkin.to_file(output_file=output_path+'profile_user.html')
  # #Review
  df_sampled=df_review.sample(frac=sample/10, random_state=seed)
  profile_checkin = ProfileReport(df_sampled)
  profile_checkin.to_file(output_file=output_path+'profile_review.html')

# Modelo SVD factorizado

In [7]:
reader = Reader( rating_scale = ( 1, 5 ) )
data = Dataset.load_from_df(df_review[['user_id', 'business_id', 'stars']], reader)
train_set, test_set = train_test_split(data, test_size=0.2, random_state=seed)
print("# user_id:", train_set.n_users,' '*5,"#business_id :", train_set.n_items,' '*5,"# reviews:", train_set.n_ratings,' '*5,"# test reviews:", len(test_set))

# user_id: 1222148       #business_id : 150007       # reviews: 3920308       # test reviews: 980077


In [29]:
search=False
if search:
  param_grid = {
      'n_factors': [25,50,100,150],
      'n_epochs': [30, 50],
      'lr_all': [0.005],
      'reg_all': [0.05],
      'random_state': [seed],
  }

## SVD

### GS

In [30]:
if search:
  gs_svd = GridSearchCV(SVD, param_grid, measures=['rmse','mae'], cv=3, n_jobs=-1)

In [31]:
%%time
if search:
  start_time = timer(None)
  gs_svd.fit(data)
  timer(start_time)


 Time taken: 0 hour(s) 2 minute(s) and 30.48 second(s).
CPU times: user 1min 41s, sys: 3.26 s, total: 1min 44s
Wall time: 2min 30s


In [32]:
if search:
  gs_svd_results=pd.DataFrame(gs_svd.cv_results).sort_values(by='rank_test_rmse')
  gs_svd_results.drop(columns='params').head(10)

,split0_test_rmse,split1_test_rmse,split2_test_rmse,mean_test_rmse,std_test_rmse,rank_test_rmse,split0_test_mae,split1_test_mae,split2_test_mae,mean_test_mae,std_test_mae,rank_test_mae,mean_fit_time,std_fit_time,mean_test_time,std_test_time,param_n_factors,param_n_epochs,param_lr_all,param_reg_all,param_random_state
3,1.341,1.340,1.342,1.341,0.001,1,1.079,1.078,1.079,1.079,0.001,1,8.569,0.757,2.786,0.094,25,50,0.005,0.050,10
1,1.342,1.341,1.342,1.342,0.000,2,1.092,1.091,1.092,1.091,0.001,8,4.439,0.427,2.432,0.253,25,30,0.005,0.050,10
7,1.342,1.342,1.343,1.342,0.000,3,1.082,1.081,1.082,1.082,0.000,3,15.267,0.587,2.923,0.035,50,50,0.005,0.050,10
5,1.343,1.342,1.343,1.343,0.000,4,1.094,1.093,1.094,1.093,0.001,10,8.372,0.489,2.702,0.201,50,30,0.005,0.050,10
0,1.343,1.342,1.344,1.343,0.001,5,1.092,1.090,1.091,1.091,0.001,7,3.561,0.114,2.311,0.159,25,30,0.005,0.020,10
11,1.345,1.344,1.345,1.344,0.000,6,1.088,1.086,1.087,1.087,0.001,5,22.617,0.454,3.474,0.168,100,50,0.005,0.050,10
4,1.345,1.345,1.345,1.345,0.000,7,1.094,1.093,1.094,1.094,0.001,11,8.304,0.402,2.396,0.064,50,30,0.005,0.020,10
9,1.345,1.345,1.346,1.345,0.000,8,1.098,1.096,1.097,1.097,0.001,13,12.921,0.503,4.260,0.567,100,30,0.005,0.050,10
2,1.346,1.346,1.348,1.347,0.001,9,1.082,1.081,1.082,1.081,0.001,2,7.158,0.644,2.420,0.274,25,50,0.005,0.020,10
15,1.347,1.346,1.347,1.347,0.000,10,1.093,1.090,1.092,1.092,0.001,9,23.910,2.942,2.299,0.030,150,50,0.005,0.050,10


In [33]:
if search:
  best_params = gs_svd.best_params['rmse']
  best_score  = gs_svd.best_score['rmse']

  # Ver los mejores parámetros
  print("Mejores parámetros encontrados: ", best_params)
  print("Mejor RMSE: ", best_score)

Mejores parámetros encontrados:  {'n_factors': 25, 'n_epochs': 50, 'lr_all': 0.005, 'reg_all': 0.05, 'random_state': 10}
Mejor RMSE:  1.340910606191147


In [34]:
svd_best_model = gs_svd.best_estimator['rmse']

In [35]:
%%time
if search:
  svd_best_model.fit(train_set)

  # Evaluar modelo en test
  print('Evaluación de Test')
  predictions = svd_best_model.test(test_set)
  rmse = accuracy.rmse(predictions)
  mae = accuracy.mae(predictions)

Evaluación de Test
RMSE: 1.3295
MAE:  1.0660
CPU times: user 30.1 s, sys: 3.36 ms, total: 30.1 s
Wall time: 30.1 s


### Final model

In [8]:
# best params
%%time
final_svd=SVD(
    n_factors = 25,
    n_epochs = 30,
    lr_all = 0.005,
    reg_all = 0.05,
    random_state = seed,
)
final_svd.fit(train_set)

print('Evaluación de Test')
predictions = final_svd.test(test_set)
rmse = accuracy.rmse(predictions)
mae = accuracy.mae(predictions)

Evaluación de Test
RMSE: 1.2793
MAE:  1.0221
CPU times: user 2min 22s, sys: 0 ns, total: 2min 22s
Wall time: 2min 22s


### Guardar modelo

In [9]:
# Guardar modelo
svd_saved_path='/content/drive/MyDrive/Maestria Andes/Sistemas_recomendacion__/Taller 2/models/'
joblib.dump(final_svd,svd_saved_path+'model_SVD_07.joblib')

['/content/drive/MyDrive/Maestria Andes/Sistemas_recomendacion__/Taller 2/models/model_SVD_07.joblib']

## SVD++

### GS

In [37]:
if search:
  gs_svdpp = GridSearchCV(SVDpp, param_grid, measures=['rmse','mae'], cv=3, n_jobs=-1)

In [38]:
%%time
if search:
  start_time = timer(None)
  gs_svdpp.fit(data)
  timer(start_time)


 Time taken: 0 hour(s) 34 minute(s) and 59.82 second(s).
CPU times: user 1min 50s, sys: 10.3 s, total: 2min
Wall time: 34min 59s


In [39]:
if search:
  gs_svdpp_results=pd.DataFrame(gs_svdpp.cv_results).sort_values(by='rank_test_rmse')
  gs_svdpp_results.drop(columns='params').head(10)

,split0_test_rmse,split1_test_rmse,split2_test_rmse,mean_test_rmse,std_test_rmse,rank_test_rmse,split0_test_mae,split1_test_mae,split2_test_mae,mean_test_mae,std_test_mae,rank_test_mae,mean_fit_time,std_fit_time,mean_test_time,std_test_time,param_n_factors,param_n_epochs,param_lr_all,param_reg_all,param_random_state
1,1.346,1.343,1.347,1.345,0.002,1,1.098,1.094,1.097,1.096,0.001,3,88.490,1.498,17.811,0.507,25,30,0.005,0.050,10
3,1.350,1.347,1.351,1.350,0.002,2,1.090,1.087,1.090,1.089,0.002,1,133.071,1.619,16.740,0.150,25,50,0.005,0.050,10
5,1.351,1.348,1.351,1.350,0.002,3,1.103,1.100,1.103,1.102,0.001,6,152.034,1.398,17.170,0.593,50,30,0.005,0.050,10
0,1.351,1.348,1.352,1.351,0.002,4,1.102,1.098,1.101,1.100,0.002,5,86.158,1.816,18.119,0.391,25,30,0.005,0.020,10
7,1.354,1.351,1.354,1.353,0.002,5,1.096,1.094,1.096,1.095,0.001,2,240.437,4.398,18.645,0.827,50,50,0.005,0.050,10
9,1.358,1.355,1.358,1.357,0.002,6,1.110,1.108,1.111,1.110,0.001,11,298.659,2.313,19.124,0.298,100,30,0.005,0.050,10
4,1.358,1.355,1.358,1.357,0.002,7,1.108,1.106,1.109,1.108,0.001,9,140.432,0.538,18.421,1.772,50,30,0.005,0.020,10
11,1.358,1.355,1.359,1.358,0.002,8,1.103,1.101,1.105,1.103,0.001,7,450.047,5.529,17.559,0.403,100,50,0.005,0.050,10
2,1.362,1.359,1.363,1.361,0.002,9,1.099,1.096,1.099,1.098,0.001,4,140.848,3.520,16.672,0.455,25,50,0.005,0.020,10
15,1.363,1.360,1.364,1.362,0.002,10,1.111,1.108,1.111,1.110,0.001,10,572.106,40.657,11.518,0.153,150,50,0.005,0.050,10


In [40]:
if search:
  best_params = gs_svdpp.best_params['rmse']
  best_score  = gs_svdpp.best_score['rmse']

  # Ver los mejores parámetros
  print("Mejores parámetros encontrados: ", best_params)
  print("Mejor RMSE: ", best_score)

Mejores parámetros encontrados:  {'n_factors': 25, 'n_epochs': 30, 'lr_all': 0.005, 'reg_all': 0.05, 'random_state': 10}
Mejor RMSE:  1.3453655321428062


In [41]:
%%time
if search:
  svdpp_best_model = gs_svdpp.best_estimator['rmse']
  svdpp_best_model.fit(train_set)

  # Evaluar modelo en test
  print('Evaluación de Test')
  predictions = svdpp_best_model.test(test_set)
  rmse = accuracy.rmse(predictions)
  mae = accuracy.mae(predictions)

RMSE: 1.3351
MAE:  1.0849
Modelo SVD++ - RMSE: 1.335092549672741, MAE: 1.0849460366627879
CPU times: user 1min 52s, sys: 13.8 ms, total: 1min 52s
Wall time: 1min 52s


### Final model

In [10]:
# best params
%%time
final_svdpp=SVDpp(
    n_factors = 25,
    n_epochs = 30,
    lr_all = 0.005,
    reg_all = 0.05,
    random_state = seed,
)
final_svdpp.fit(train_set)

print('Evaluación de Test')
predictions = final_svdpp.test(test_set)
rmse = accuracy.rmse(predictions)
mae = accuracy.mae(predictions)

Evaluación de Test
RMSE: 1.2864
MAE:  1.0312
CPU times: user 18min 24s, sys: 2.11 s, total: 18min 26s
Wall time: 18min 27s


### Guardar modelo

In [11]:
# Guardar modelo
svdpp_saved_path='/content/drive/MyDrive/Maestria Andes/Sistemas_recomendacion__/Taller 2/models/'
joblib.dump(final_svdpp,svdpp_saved_path+'model_SVDpp_07.joblib')

['/content/drive/MyDrive/Maestria Andes/Sistemas_recomendacion__/Taller 2/models/model_SVDpp_07.joblib']

# Load of models

In [ ]:
load_model=joblib.load(svdpp_saved_path+'model_SVDpp.joblib')

In [ ]:
print('(n_users, n_factors)',load_model.pu.shape)
print('(n_items, n_factors)',load_model.qi.shape)
print('Parametros:',
  '\n  n_factors:', load_model.n_factors,
  '\n  n_epochs:', load_model.n_epochs,
      )

# End